# 01-sparksession-config

01-sparksession-config/01-sparksession-and-config.py
Demonstrates SparkSession creation, configuration, catalog, and session hierarchy.

Patterns covered:
  1.  SparkSession creation — version, appName, master
  2.  Reading spark configs
  3.  Overriding a config at runtime
  4.  sparkContext attributes
  5.  SparkSession vs SparkContext vs SQLContext hierarchy
  6.  getOrCreate idempotency
  7.  Creating a DataFrame from the session
  8.  Running Spark SQL inline
  9.  spark.catalog — listDatabases / listTables
  10. SparkConf — getAll, filter for spark.sql
  11. sc.uiWebUrl
  12. Builder chain (commented example)

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath(__file__)), '..', '00-setup'))
from spark_session import get_spark, output_path, stop_and_wait
from seed_data import load_all, register_views

from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window

## 1. SparkSession creation

In [ ]:
spark = get_spark("01-sparksession-config")

print("=" * 60)
print("1. SparkSession creation")
print("=" * 60)
print(f"   spark.version              : {spark.version}")
print(f"   sparkContext.appName       : {spark.sparkContext.appName}")
print(f"   sparkContext.master        : {spark.sparkContext.master}")

## 2. Reading spark configs

In [ ]:
print("\n" + "=" * 60)
print("2. Reading spark configs")
print("=" * 60)
shuffle_partitions = spark.conf.get("spark.sql.shuffle.partitions")
executor_memory    = spark.conf.get("spark.executor.memory", "not set")
print(f"   spark.sql.shuffle.partitions : {shuffle_partitions}")
print(f"   spark.executor.memory        : {executor_memory}")

## 3. Overriding a config at runtime

In [ ]:
print("\n" + "=" * 60)
print("3. Overriding a config at runtime")
print("=" * 60)
print(f"   Before: spark.sql.shuffle.partitions = {spark.conf.get('spark.sql.shuffle.partitions')}")
spark.conf.set("spark.sql.shuffle.partitions", "8")
print(f"   After : spark.sql.shuffle.partitions = {spark.conf.get('spark.sql.shuffle.partitions')}")
# Reset to original for the rest of the script
spark.conf.set("spark.sql.shuffle.partitions", "4")

## 4. sparkContext attributes

In [ ]:
print("\n" + "=" * 60)
print("4. sparkContext attributes")
print("=" * 60)
sc = spark.sparkContext
print(f"   sc.defaultParallelism : {sc.defaultParallelism}")
print(f"   sc.appName            : {sc.appName}")
print(f"   sc.master             : {sc.master}")

## 5. SparkSession vs SparkContext vs SQLContext — type hierarchy

In [ ]:
print("\n" + "=" * 60)
print("5. SparkSession / SparkContext / SQLContext hierarchy")
print("=" * 60)
# SQLContext wraps SparkContext; SparkSession supersedes both.
# SparkSession.builder.getOrCreate() always returns or reuses the session.
print(f"   type(spark)                : {type(spark)}")
print(f"   type(spark.sparkContext)   : {type(spark.sparkContext)}")
# SQLContext is legacy — obtain it from the existing session
from pyspark.sql import SQLContext
sql_ctx = SQLContext.getOrCreate(sc)
print(f"   type(SQLContext.getOrCreate): {type(sql_ctx)}")
print("   Hierarchy: SparkSession ⊃ SQLContext ⊃ SparkContext")
print("   SparkSession is the unified entry point since Spark 2.0.")

## 6. getOrCreate idempotency — calling get_spark() twice returns the SAME session

In [ ]:
print("\n" + "=" * 60)
print("6. getOrCreate pattern — same session returned")
print("=" * 60)
spark2 = get_spark("01-sparksession-config")
print(f"   spark  id : {id(spark)}")
print(f"   spark2 id : {id(spark2)}")
print(f"   spark is spark2 : {spark is spark2}")
# getOrCreate returns the existing session regardless of appName passed

## 7. Creating a DataFrame from the session

In [ ]:
print("\n" + "=" * 60)
print("7. Creating a DataFrame from the session")
print("=" * 60)
inline_df = spark.createDataFrame([(1, 'a'), (2, 'b')], ['id', 'val'])
inline_df.show()
inline_df.printSchema()

## 8. Running Spark SQL

In [ ]:
print("\n" + "=" * 60)
print("8. Running Spark SQL")
print("=" * 60)
spark.sql("SELECT 1+1 AS result").show()

## 9. spark.catalog — listDatabases and listTables

In [ ]:
print("\n" + "=" * 60)
print("9. spark.catalog — before and after register_views()")
print("=" * 60)
print("   Databases available:")
for db in spark.catalog.listDatabases():
    print(f"     {db.name}  (location: {db.locationUri})")

print("\n   Tables before register_views():")
tables_before = spark.catalog.listTables()
print(f"     count = {len(tables_before)}")

dfs = register_views(spark)

print("\n   Tables after register_views():")
for tbl in spark.catalog.listTables():
    print(f"     {tbl.name:25s}  tableType={tbl.tableType}  isTemporary={tbl.isTemporary}")

## 10. SparkConf — getAll, filtered for spark.sql.*

In [ ]:
print("\n" + "=" * 60)
print("10. SparkConf — all configs filtered for 'spark.sql'")
print("=" * 60)
all_conf = sc.getConf().getAll()
sql_conf = [(k, v) for k, v in sorted(all_conf) if "spark.sql" in k]
for k, v in sql_conf:
    print(f"   {k:<50s} = {v}")

## 11. Spark UI URL

In [ ]:
print("\n" + "=" * 60)
print("11. Spark UI URL")
print("=" * 60)
print(f"   sc.uiWebUrl : {sc.uiWebUrl}")

## 12. SparkSession.builder pattern (commented — do NOT create a second session)

In [ ]:
print("\n" + "=" * 60)
print("12. Full builder chain (commented example — not executed)")
print("=" * 60)
print("""
   # Full builder chain:
   # spark = (
   #     SparkSession.builder
   #     .appName("my-app")
   #     .master("local[*]")
   #     .config("spark.sql.shuffle.partitions", "4")
   #     .config("spark.driver.memory", "2g")
   #     .config("spark.ui.enabled", "true")
   #     .config("spark.sql.adaptive.enabled", "true")
   #     .enableHiveSupport()          # only if Hive metastore is present
   #     .getOrCreate()
   # )
   #
   # .getOrCreate() returns an existing session if one is already running
   # (same JVM). Use this pattern in every script so the session is reused
   # when running interactively or inside notebooks.
""")

## Row-count verification for all 9 tables

In [ ]:
print("\n" + "=" * 60)
print("Row counts for all 9 registered tables")
print("=" * 60)
counts = {name: df.count() for name, df in dfs.items()}
for name, cnt in counts.items():
    print(f"   {name:<25s} : {cnt}")

stop_and_wait(spark)